In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'input.txt.1'
87input.txt.1           45% [============>                 ]  192.00K    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87input.txt.1          100% [=============================>]  424.87K    1.94MB/s87HTTP response 200  [https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt]
87input.txt.1          100% [=============================>]  424.87K    1.94MB/s87[Files: 1  Bytes: 424.87K [380.]8

In [2]:
import numpy as np 
import pandas as pd
import math 
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
from torchinfo import summary
from tqdm import tqdm 
import matplotlib.pyplot as plt 

In [3]:
from transformer import LanguageModelingDataset, Transformer
from train import train_model
from eval import evaluate_model

In [4]:
# Hyperparameters
lr = 3e-3
dmodel = 128
context_len = 128
batch_size = 32
num_heads = 6
num_layers = 6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 30

#paths
file_path = 'input.txt'

In [5]:
bpe_tokenizer = tiktoken.encoding_for_model('gpt-2')

train_tokens = LanguageModelingDataset(file_path, bpe_tokenizer, context_len-1, 'train')
valid_tokens = LanguageModelingDataset(file_path, bpe_tokenizer, context_len-1, 'valid')

train_dataloader = DataLoader(
    dataset= train_tokens,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

valid_dataloader = DataLoader(
    dataset= valid_tokens,
    batch_size=batch_size,
    num_workers=0,
    drop_last=True
)

In [6]:
transformer = Transformer(num_layers, context_len, dmodel, num_heads, bpe_tokenizer.n_vocab, dropout).to(device)
optimizer = optim.Adagrad(params = transformer.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()
summary(transformer)

Layer (type:depth-idx)                        Param #
Transformer                                   --
├─Embedding: 1-1                              6,432,896
├─PositionalEncoding: 1-2                     --
├─Linear: 1-3                                 6,432,896
├─Sequential: 1-4                             --
│    └─Block: 2-1                             --
│    │    └─MultiHeadAttention: 3-1           81,272
│    │    └─LayerNorm: 3-2                    256
│    │    └─LayerNorm: 3-3                    256
│    │    └─FFN: 3-4                          131,712
│    └─Block: 2-2                             --
│    │    └─MultiHeadAttention: 3-5           81,272
│    │    └─LayerNorm: 3-6                    256
│    │    └─LayerNorm: 3-7                    256
│    │    └─FFN: 3-8                          131,712
│    └─Block: 2-3                             --
│    │    └─MultiHeadAttention: 3-9           81,272
│    │    └─LayerNorm: 3-10                   256
│    │    └─LayerNorm: 

In [7]:
results = {
    'train_loss' : [],
    'eval_loss' : [],
    'train_acc' : [],
    'eval_acc' : [],
    'perplexity' : []
}

for epoch in tqdm(range(EPOCHS)):
    tl, ta = train_model(transformer, train_dataloader, loss_fn, optimizer, epoch)
    el, ea, per = evaluate_model(transformer, valid_dataloader, loss_fn, epoch)
    results['train_loss'].append(tl)
    results['eval_loss'].append(el)
    results['train_acc'].append(ta)
    results['eval_acc'].append(ea)
    results['perplexity'].append(per)

  0%|          | 0/30 [00:07<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Define the order of plotting and pretty titles
keys = ['train_loss', 'eval_loss', 'train_acc', 'eval_acc', 'perplexity']
titles = ['Train Loss', 'Eval Loss', 'Train Accuracy', 'Eval Accuracy', 'Perplexity']

# Create a 2x3 grid (2 rows, 3 columns)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Flatten the 2D array of axes into a 1D array for easy looping
axes = axes.flatten()

for i, (key, title) in enumerate(zip(keys, titles)):
    ax = axes[i]
    
    # Plot the data (added small markers for easier visual tracking of epochs)
    ax.plot(results[key], marker='.', markersize=4, linewidth=1.5)
    
    # Formatting
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epochs / Steps')
    ax.set_ylabel(title)
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Optional: Highlight the final value
    if len(results[key]) > 0:
        final_val = results[key][-1]
        ax.annotate(f'{final_val:.4f}', 
                    xy=(len(results[key])-1, final_val), 
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=9, color='red')

# Hide the 6th empty subplot
axes[-1].set_visible(False)

# Adjust layout to prevent overlapping labels
plt.tight_layout()
plt.show()

In [ ]:
query = torch.tensor(bpe_tokenizer.encode('What, have you forgot me?')).unsqueeze(0).to(device)
gen = transformer.generate(query, 120, decoding='top-k', k=10)
"".join((bpe_tokenizer.decode(gen.tolist()[0])))